In [1]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split, StratifiedKFold
from torchvision import transforms

In [2]:
X = np.load("X_preprocessed_conf1405.npy")
y_encoded = np.load("y_labels_conf1405.npy")

In [3]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Full dataset shape:", X.shape)
print("TrainVal data shape:", X_trainval.shape)
print("Test data shape:", X_test.shape)
print("TrainVal labels shape:", y_trainval.shape)
print("Test labels shape:", y_test.shape)

Full dataset shape: (1618, 1, 224, 224)
TrainVal data shape: (1294, 1, 224, 224)
Test data shape: (324, 1, 224, 224)
TrainVal labels shape: (1294,)
Test labels shape: (324,)


In [4]:
#Create Test Dataset and Test Loader

batch_size = 16

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Test tensor shape:", X_test_tensor.shape)
print("Test labels shape:", y_test_tensor.shape)
print("Number of test batches:", len(test_loader))

test_images, test_labels = next(iter(test_loader))
print("One test batch image shape:", test_images.shape)
print("One test batch label shape:", test_labels.shape)

Test tensor shape: torch.Size([324, 1, 224, 224])
Test labels shape: torch.Size([324])
Number of test batches: 21
One test batch image shape: torch.Size([16, 1, 224, 224])
One test batch label shape: torch.Size([16])


In [5]:
# Define Stratified 5-Fold Cross Validation

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [6]:
# Define Training Augmentation

train_augmentation = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    )
])

In [7]:
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm
import os

In [8]:
def prepare_for_convnext(images):

    # Convert grayscale 1-channel OCT image to 3-channel
    if images.shape[1] == 1:
        images = images.repeat(1, 3, 1, 1)

    # ImageNet normalization for pretrained ConvNeXt
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

    images = (images - mean) / std

    return images

In [9]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(
            inputs,
            targets,
            weight=self.alpha,
            reduction="none"
        )

        pt = torch.exp(-ce_loss)

        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        else:
            return focal_loss

In [12]:
# Create Train and Validation DataLoaders for Each Fold

for fold, (train_index, val_index) in enumerate(skf.split(X_trainval, y_trainval), 1):

    # Split TrainVal data into training and validation for this fold
    X_train = X_trainval[train_index]
    X_val = X_trainval[val_index]

    y_train = y_trainval[train_index]
    y_val = y_trainval[val_index]

    # Convert numpy arrays to PyTorch tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)

    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val, dtype=torch.long)

    # Apply augmentation only to training images
    augmented_images = []

    for image in X_train_tensor:
        augmented_image = train_augmentation(image)
        augmented_images.append(augmented_image)

    augmented_images = torch.stack(augmented_images)

    # Combine original + augmented training images
    X_train_final = torch.cat((X_train_tensor, augmented_images), dim=0)

    # Duplicate labels for augmented images
    y_train_final = torch.cat((y_train_tensor, y_train_tensor), dim=0)

    # Create TensorDataset
    train_dataset = TensorDataset(X_train_final, y_train_final)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

    # WeightedRandomSampler
    class_counts = np.bincount(y_train_final.numpy())

    class_weights = 1.0 / class_counts
    class_weights = class_weights / class_weights.sum() * len(class_counts)
    class_weights = torch.tensor(class_weights, dtype=torch.float32)

    # Create DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    print("Training ConvNeXt-Tiny for this fold...")

    # Create new pretrained ConvNeXt-Tiny model for this fold
    weights = ConvNeXt_Tiny_Weights.DEFAULT
    model = convnext_tiny(weights=weights)

    # Freeze feature extractor because CPU training is slow
    for param in model.features.parameters():
        param.requires_grad = False

    # Unfreeze last ConvNeXt block
    for param in model.features[-1].parameters():
        param.requires_grad = True    

    num_classes = len(np.unique(y_encoded))
    print("Number of classes:", num_classes)

    # Change final layer according to your number of classes
    model.classifier[2] = nn.Linear(
        model.classifier[2].in_features,
        num_classes
    )

    criterion = FocalLoss(alpha=class_weights, gamma=2.0)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.00005,
        weight_decay=1e-4
    )

    num_epochs = 27

    start_epoch = 0
    best_val_acc = 0
    
    latest_checkpoint_path = f"convnext_fold_{fold}_latest_checkpoint2.pth"
    best_checkpoint_path = f"convnext_fold_{fold}_best_checkpoint2.pth"
    old_checkpoint_path = f"convnext_fold_{fold}_checkpoint2.pth"

    checkpoint_loaded = False

    # 1. First priority: latest checkpoint
    if os.path.exists(latest_checkpoint_path):
        checkpoint = torch.load(latest_checkpoint_path, map_location="cpu")
    
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    
        start_epoch = checkpoint["epoch"] + 1
        best_val_acc = checkpoint.get("best_val_acc", 0.0)
    
        checkpoint_loaded = True
        print(f"Resuming Fold {fold} from latest checkpoint at Epoch {start_epoch + 1}")

    # 2. Second priority: old checkpoint
    elif os.path.exists(old_checkpoint_path):
        checkpoint = torch.load(old_checkpoint_path, map_location="cpu")
    
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    
        start_epoch = checkpoint["epoch"] + 1
        best_val_acc = checkpoint.get("best_val_acc", 0.0)
    
        checkpoint_loaded = True
        print(f"Resuming Fold {fold} from old checkpoint at Epoch {start_epoch + 1}")
        
    # 3. Fresh training
    else:
        print(f"Starting Fold {fold} from Epoch 1")

    # 4. Create best checkpoint only if checkpoint was loaded and best file is missing
    if checkpoint_loaded and not os.path.exists(best_checkpoint_path):
        torch.save({
            "fold": fold,
            "epoch": start_epoch - 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_acc": best_val_acc
        }, best_checkpoint_path)
    
        print(f"Best checkpoint file created for Fold {fold}")    
    
    for epoch in range(start_epoch, num_epochs):
    
        model.train()
    
        train_loss = 0.0
        train_correct = 0
        train_total = 0
    
        for images, labels in tqdm(train_loader):
    
            images = prepare_for_convnext(images)
    
            optimizer.zero_grad()
    
            outputs = model(images)
            loss = criterion(outputs, labels)
    
            loss.backward()
            optimizer.step()
    
            train_loss += loss.item()
    
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
    
        train_loss = train_loss / len(train_loader)
        train_acc = 100 * train_correct / train_total
    
    
        model.eval()
    
        val_loss = 0.0
        val_correct = 0
        val_total = 0
    
        with torch.no_grad():
            for images, labels in val_loader:
    
                images = prepare_for_convnext(images)
    
                outputs = model(images)
                loss = criterion(outputs, labels)
    
                val_loss += loss.item()
    
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
    
        val_loss = val_loss / len(val_loader)
        val_acc = 100 * val_correct / val_total

        print(
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f} "
            f"Train Acc: {train_acc:.2f}% "
            f"Val Loss: {val_loss:.4f} "
            f"Val Acc: {val_acc:.2f}%"
        )
        
        # Save best checkpoint only when validation accuracy improves
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        
            torch.save({
                "fold": fold,
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_acc": best_val_acc
            }, best_checkpoint_path)
        
            print(f"Best checkpoint saved for Fold {fold}, Epoch {epoch + 1}")
        
        # Save latest checkpoint after every epoch for resume training
        torch.save({
            "fold": fold,
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_acc": best_val_acc
        }, latest_checkpoint_path)
        
        print(f"Latest checkpoint saved for Fold {fold}, Epoch {epoch + 1}")

Training ConvNeXt-Tiny for this fold...
Number of classes: 7
Resuming Fold 1 from Epoch 26
Resuming Fold 1 from old checkpoint at Epoch 20
Starting Fold 1 from Epoch 1


100%|████████████████████████████████████████████████████████████████████████████████| 130/130 [04:29<00:00,  2.07s/it]


Epoch [20/27] Train Loss: 0.0128 Train Acc: 74.11% Val Loss: 0.0867 Val Acc: 64.86%
Latest checkpoint saved for Fold 1, Epoch 20


100%|████████████████████████████████████████████████████████████████████████████████| 130/130 [03:46<00:00,  1.74s/it]


Epoch [21/27] Train Loss: 0.0127 Train Acc: 75.85% Val Loss: 0.1320 Val Acc: 64.86%
Latest checkpoint saved for Fold 1, Epoch 21


100%|████████████████████████████████████████████████████████████████████████████████| 130/130 [03:40<00:00,  1.69s/it]


... (45 outputs left out)

100%|████████████████████████████████████████████████████████████████████████████████| 130/130 [03:25<00:00,  1.58s/it]


Epoch [26/27] Train Loss: 0.0147 Train Acc: 78.28% Val Loss: 0.0901 Val Acc: 75.97%
Latest checkpoint saved for Fold 5, Epoch 26


100%|████████████████████████████████████████████████████████████████████████████████| 130/130 [03:21<00:00,  1.55s/it]


Epoch [27/27] Train Loss: 0.0064 Train Acc: 80.55% Val Loss: 0.0913 Val Acc: 77.91%
Latest checkpoint saved for Fold 5, Epoch 27


In [10]:
import torch
import torch.nn as nn
import numpy as np
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = len(np.unique(y_encoded))

checkpoint_paths = [
    "convnext_fold_1_best_checkpoint2.pth",
    "convnext_fold_2_best_checkpoint2.pth",
    "convnext_fold_3_best_checkpoint2.pth",
    "convnext_fold_4_best_checkpoint2.pth",
    "convnext_fold_5_best_checkpoint2.pth"
]

models = []

for path in checkpoint_paths:
    weights = ConvNeXt_Tiny_Weights.DEFAULT
    model = convnext_tiny(weights=weights)

    model.classifier[2] = nn.Linear(
        model.classifier[2].in_features,
        num_classes
    )

    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    model = model.to(device)
    model.eval()

    models.append(model)

print("All 5 fold models loaded successfully")

All 5 fold models loaded successfully


In [11]:
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader):

        images = prepare_for_convnext(images)
        images = images.to(device)
        labels = labels.to(device)

        fold_probs = []

        for model in models:
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            fold_probs.append(probs)

        # Average probabilities from 5 fold models
        avg_probs = torch.mean(torch.stack(fold_probs), dim=0)

        # Final prediction
        _, predicted = torch.max(avg_probs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

100%|██████████████████████████████████████████████████████████████████████████████████| 21/21 [03:18<00:00,  9.46s/it]


In [12]:
class_names = ["AMD", "DME", "ERM", "NO", "RAO", "RVO", "VID"]

In [21]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

accuracy = accuracy_score(all_labels, all_preds)

precision_macro = precision_score(all_labels, all_preds, average="macro", zero_division=0)
recall_macro = recall_score(all_labels, all_preds, average="macro", zero_division=0)
f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)

precision_weighted = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
recall_weighted = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
f1_weighted = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

print("========== Final Test Evaluation ==========")
print(f"Accuracy           : {accuracy:.4f}")
print(f"Macro Precision    : {precision_macro:.4f}")
print(f"Macro Recall       : {recall_macro:.4f}")
print(f"Macro F1-score     : {f1_macro:.4f}")
print(f"Weighted Precision : {precision_weighted:.4f}")
print(f"Weighted Recall    : {recall_weighted:.4f}")
print(f"Weighted F1-score  : {f1_weighted:.4f}")

========== Final Test Evaluation ==========
Accuracy           : 0.8272
Macro Precision    : 0.7991
Macro Recall       : 0.8303
Macro F1-score     : 0.8056
Weighted Precision : 0.8583
Weighted Recall    : 0.8272
Weighted F1-score  : 0.8321


In [19]:
accuracy = accuracy_score(all_labels, all_preds)

print("Ensemble Test Accuracy:", accuracy * 100)

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

Ensemble Test Accuracy: 82.71604938271605

Classification Report:
              precision    recall  f1-score   support

         AMD       0.95      0.80      0.87       177
         DME       0.68      0.90      0.78        29
         ERM       0.91      0.74      0.82        27
          NO       0.65      0.93      0.76        57
         RAO       0.75      0.75      0.75         4
         RVO       0.88      0.79      0.83        19
         VID       0.77      0.91      0.83        11

    accuracy                           0.83       324
   macro avg       0.80      0.83      0.81       324
weighted avg       0.86      0.83      0.83       324


Confusion Matrix:
[[141   7   1  22   1   2   3]
 [  1  26   0   2   0   0   0]
 [  3   1  20   3   0   0   0]
 [  2   1   1  53   0   0   0]
 [  0   0   0   1   3   0   0]
 [  1   3   0   0   0  15   0]
 [  0   0   0   1   0   0  10]]
